### Pacotes e Ambiente

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
import os, sys, shutil, time

In [2]:
# check environment variables
print("Python:", sys.executable)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("java:", shutil.which("java"))

Python: c:\Users\clamo\Documents\Doutorado\HC\hc_models\.venv\Scripts\python.exe
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.x-hotspot
java: C:\Program Files\Eclipse Adoptium\jdk-17.0.16.8-hotspot\bin\java.EXE


### Iniciar Spark Session

In [3]:
# init spark

spark = (SparkSession.builder
         .appName("notebook")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions","8")
         .getOrCreate()
         )

spark.version

KeyboardInterrupt: 

### Ler os dados

In [ ]:
# ler dados de exames
exames = spark.read.option("header", True).csv("exportados_20250902_165405/paciente.csv")

### Pré-processamento

In [ ]:
# dicionarios de valores e categorias
map_cor = {
    "B": "cor_branca",
    "I": "cor_indigena",
    "A": "cor_amarela",
    "P": "cor_preta",
    "M": "cor_marrom",
    "O": "cor_outras"
}

# cria colunas de cor binarias, 1 se for o codigo, senao 0
for codigo, nome_coluna in map_cor.items():
    exames = exames.withColumn(nome_coluna, F.when(F.col("Cor") == codigo, 1).otherwise(0))


# cria colunas de sexo binarias
exames = exames.withColumn("sexo_feminino", F.when(F.col("Sexo") == "Feminino", 1).otherwise(0))
exames = exames.withColumn("sexo_masculino", F.when(F.col("Sexo") == "Masculino", 1).otherwise(0))

# renomeia coluna de idade
exames = exames.withColumnRenamed("Idade", "idade")

# selecionar colunas de interesse
exames = exames.select(
    "prontuario",
    "idade",
    "cor_branca",
    "cor_indigena",
    "cor_amarela",
    "cor_preta",
    "cor_marrom",
    "cor_outras",
    "sexo_feminino",
    "sexo_masculino"
)

# salvar dados preprocessados
exames.write.mode("overwrite").parquet("dados_preprocessados/features_paciente.parquet")